# Abnormal Volume and Volume–Momentum Interaction

## Research question

Does unusual trading participation contain useful cross-sectional information beyond the retained factors?

Two signals are considered:

1. **Abnormal volume:** current log dollar volume relative to the stock's own trailing 21-day history.

$$
AV_{i,t}
=
\log(DV_{i,t})
- \frac{1}{21}
\sum_{s=1}^{21}\log(DV_{i,t-s}).
$$

2. **Volume–momentum interaction:** abnormal volume interacted with conventional 12–1 momentum.

$$
VM_{i,t}
=
MOM^{z}_{i,t} \times AV^{z}_{i,t}.
$$

The abnormal-volume calculation uses only information available through date $t$. Its trailing reference level is shifted by one day, ensuring that the current observation is not included in its own benchmark.

The initial stage evaluates construction validity, coverage, distributions, and similarity to existing factors. Predictive IC tests will follow only after these checks pass.

## 1. Setup and data audit

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from alpha_research.data_loader import load_parquet
from alpha_research.signal_processing import (
    add_sector_neutral_factor,
    process_factor,
)

In [2]:
current = Path.cwd().resolve()
for _ in range(10):
    if (current / "pyproject.toml").exists():
        ROOT = current
        break
    current = current.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

factor_panel_path = (
    ROOT / "data" / "processed" / "factor_panel.parquet"
)

assert factor_panel_path.exists(), (
    f"Factor panel not found: {factor_panel_path}"
)

volume_panel = load_parquet(factor_panel_path).copy()

volume_panel["date"] = pd.to_datetime(
    volume_panel["date"]
).dt.tz_localize(None)

volume_panel = (
    volume_panel
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

required_columns = {
    "date",
    "ticker",
    "sector",
    "log_dollar_volume",
    "mom_12_1m_z",
    "mom_12_1m_raw",
    "realised_vol_63_z",
    "forward_ret_1d",
    "forward_ret_5d",
}

missing_columns = required_columns - set(volume_panel.columns)

assert not missing_columns, (
    f"Missing required columns: {sorted(missing_columns)}"
)

assert not volume_panel[
    ["date", "ticker"]
].duplicated().any()

volume_panel[
    [
        "date",
        "ticker",
        "log_dollar_volume",
        "mom_12_1m_z",
        "forward_ret_5d",
    ]
].head()

,date,ticker,log_dollar_volume,mom_12_1m_z,forward_ret_5d
0,2015-01-02,AAPL,22.484026,NaN,0.024512
1,2015-01-05,AAPL,22.644639,NaN,0.028235
2,2015-01-06,AAPL,22.667975,NaN,0.037267
3,2015-01-07,AAPL,22.186848,NaN,0.019025
4,2015-01-08,AAPL,22.616723,NaN,-0.045313


## 2. Signal construction

### 2.1 Abnormal volume

In [3]:
VOLUME_LOOKBACK = 21
VOLUME_MIN_PERIODS = 21

volume_panel[
    "log_dollar_volume_mean_21_lag1"
] = (
    volume_panel
    .groupby("ticker")["log_dollar_volume"]
    .transform(
        lambda series: (
            series
            .shift(1)
            .rolling(
                window=VOLUME_LOOKBACK,
                min_periods=VOLUME_MIN_PERIODS,
            )
            .mean()
        )
    )
)

volume_panel["abnormal_volume_21_raw"] = (
    volume_panel["log_dollar_volume"]
    -
    volume_panel["log_dollar_volume_mean_21_lag1"]
)

In [4]:
volume_panel = process_factor(
    volume_panel,
    raw_column="abnormal_volume_21_raw",
    output_prefix="abnormal_volume_21",
    lower_quantile=0.01,
    upper_quantile=0.99,
)

volume_panel = add_sector_neutral_factor(
    volume_panel,
    factor_column="abnormal_volume_21_winsorised",
    output_column="abnormal_volume_21_sector_neutral_z",
    sector_column="sector",
    min_sector_observations=3,
)

### 2.2 Volume–momentum interaction

In [ ]:
volume_panel["volume_momentum_interaction_raw"] = (
    volume_panel["mom_12_1m_z"] *
    volume_panel["abnormal_volume_21_z"]
)

volume_panel = process_factor(
    volume_panel,
    raw_column="volume_momentum_interaction_raw",
    output_prefix="volume_momentum_interaction",
    lower_quantile=0.01,
    upper_quantile=0.99,
)

volume_panel = add_sector_neutral_factor(
    volume_panel,
    factor_column="volume_momentum_interaction_winsorised",
    output_column="volume_momentum_interaction_sector_neutral_z",
    sector_column="sector",
    min_sector_observations=3,
)

## 3. Construction audits

### 3.1 Manual no-lookahead check

In [6]:
sample_row = (
    volume_panel
    .dropna(subset=["abnormal_volume_21_raw"])
    .iloc[len(
        volume_panel.dropna(
            subset=["abnormal_volume_21_raw"]
        )
    ) // 2]
)

sample_ticker = sample_row["ticker"]
sample_date = sample_row["date"]

sample_history = (
    volume_panel.loc[
        (volume_panel["ticker"] == sample_ticker)
        & (volume_panel["date"] < sample_date),
        ["date", "log_dollar_volume"],
    ]
    .tail(VOLUME_LOOKBACK)
)

manual_abnormal_volume = (
    sample_row["log_dollar_volume"]
    -
    sample_history["log_dollar_volume"].mean()
)

manual_abnormal_volume_check = pd.Series(
    {
        "ticker": sample_ticker,
        "signal_date": sample_date,
        "history_observations": len(sample_history),
        "history_start": sample_history["date"].min(),
        "history_end": sample_history["date"].max(),
        "current_log_dollar_volume": (
            sample_row["log_dollar_volume"]
        ),
        "manual_trailing_mean": (
            sample_history["log_dollar_volume"].mean()
        ),
        "manual_abnormal_volume": manual_abnormal_volume,
        "calculated_abnormal_volume": (
            sample_row["abnormal_volume_21_raw"]
        ),
        "absolute_difference": abs(
            manual_abnormal_volume
            - sample_row["abnormal_volume_21_raw"]
        ),
    },
    name="value",
)

manual_abnormal_volume_check

ticker                                        JNJ
signal_date                   2026-01-27 00:00:00
history_observations                           21
history_start                 2025-12-24 00:00:00
history_end                   2026-01-26 00:00:00
current_log_dollar_volume                21.32764
manual_trailing_mean                    21.144517
manual_abnormal_volume                   0.183123
calculated_abnormal_volume               0.183123
absolute_difference                           0.0
Name: value, dtype: object

### 3.2 Coverage

In [7]:
abnormal_volume_coverage = pd.Series(
    {
        "total_rows": len(volume_panel),
        "raw_abnormal_volume_count": (
            volume_panel["abnormal_volume_21_raw"].notna().sum()
        ),
        "raw_abnormal_volume_coverage": (
            volume_panel["abnormal_volume_21_raw"].notna().mean()
        ),
        "processed_abnormal_volume_count": (
            volume_panel["abnormal_volume_21_z"].notna().sum()
        ),
        "interaction_count": (
            volume_panel["volume_momentum_interaction_z"].notna().sum()
        ),
        "interaction_coverage": (
            volume_panel["volume_momentum_interaction_z"].notna().mean()
        ),
        "tickers_with_abnormal_volume": (
            volume_panel.loc[
                volume_panel["abnormal_volume_21_z"].notna(),
                "ticker",
            ].nunique()
        ),
        "dates_with_abnormal_volume": (
            volume_panel.loc[
                volume_panel["abnormal_volume_21_z"].notna(),
                "date",
            ].nunique()
        ),
        "tickers_with_interaction": (
            volume_panel.loc[
                volume_panel["volume_momentum_interaction_z"].notna(),
                "ticker",
            ].nunique()
        ),
        "dates_with_interaction": (
            volume_panel.loc[
                volume_panel["volume_momentum_interaction_z"].notna(),
                "date",
            ].nunique()
        ),
    },
    name="value",
)

abnormal_volume_coverage

total_rows                         284249.000000
raw_abnormal_volume_count          282136.000000
raw_abnormal_volume_coverage            0.992566
processed_abnormal_volume_count    282136.000000
interaction_count                  259036.000000
interaction_coverage                    0.911300
tickers_with_abnormal_volume          100.000000
dates_with_abnormal_volume           2870.000000
tickers_with_interaction              100.000000
dates_with_interaction               2639.000000
Name: value, dtype: float64

### 3.3 Distribution diagnostics

In [8]:
abnormal_volume_distribution = (
    volume_panel[
        [
            "log_dollar_volume",
            "log_dollar_volume_mean_21_lag1",
            "abnormal_volume_21_raw",
            "abnormal_volume_21_z",
            "volume_momentum_interaction_raw",
            "volume_momentum_interaction_z",
        ]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
    .T
)

abnormal_volume_distribution

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
log_dollar_volume,284249.0,2.042137e+01,0.964648,0.000000,18.635792,19.097329,19.800818,20.318202,20.871774,22.311944,23.519060,25.762668
log_dollar_volume_mean_21_lag1,282136.0,2.041982e+01,0.903946,16.339917,18.845487,19.201659,19.844006,20.318355,20.813938,22.216942,23.370285,24.800068
abnormal_volume_21_raw,282136.0,5.002706e-03,0.368494,-4.419480,-0.747872,-0.508496,-0.231086,-0.031953,0.197654,0.653804,1.124668,3.070035
abnormal_volume_21_z,282136.0,2.014752e-19,1.000002,-3.182847,-1.927895,-1.435234,-0.674680,-0.116397,0.542908,1.855071,3.013575,5.472823
volume_momentum_interaction_raw,259036.0,1.071212e-02,1.098464,-14.720549,-3.327981,-1.459518,-0.281814,0.004088,0.305545,1.413192,3.282342,23.062864
volume_momentum_interaction_z,259036.0,-1.536095e-18,1.000002,-6.646974,-3.187017,-1.628382,-0.341099,0.006854,0.359355,1.569105,3.082332,6.803122


### 3.4 Log dollar-volume check

In [11]:
volume_panel.loc[
    volume_panel["log_dollar_volume"] <= 0,
    [
        "date",
        "ticker",
        "log_dollar_volume",
        "log_dollar_volume_mean_21_lag1",
        "abnormal_volume_21_raw",
    ],
].sort_values(["ticker", "date"])

,date,ticker,log_dollar_volume,log_dollar_volume_mean_21_lag1,abnormal_volume_21_raw
17346,2015-01-02,AMD,0.0,NaN,NaN


### 3.5 Cross-sectional similarity

In [9]:
def calculate_daily_spearman(
    panel,
    left_column,
    right_column,
    minimum_observations=20,
):
    results = []

    for date, group in panel.groupby(
        "date",
        sort=True,
    ):
        valid = group[
            [left_column, right_column]
        ].dropna()

        if len(valid) < minimum_observations:
            continue

        if (
            valid[left_column].nunique() < 2
            or valid[right_column].nunique() < 2
        ):
            continue

        correlation = valid[left_column].corr(
            valid[right_column],
            method="spearman",
        )

        results.append(
            {
                "date": date,
                "correlation": correlation,
            }
        )

    return (
        pd.DataFrame(results)
        .set_index("date")["correlation"]
    )

In [10]:
similarity_specifications = {
    "Abnormal volume vs dollar-volume level": (
        "abnormal_volume_21_z",
        "log_dollar_volume",
    ),
    "Abnormal volume vs momentum": (
        "abnormal_volume_21_z",
        "mom_12_1m_z",
    ),
    "Abnormal volume vs realised volatility": (
        "abnormal_volume_21_z",
        "realised_vol_63_z",
    ),
    "Volume–momentum interaction vs momentum": (
        "volume_momentum_interaction_z",
        "mom_12_1m_z",
    ),
}

similarity_rows = []

for comparison, (
    left_column,
    right_column,
) in similarity_specifications.items():

    daily_correlation = calculate_daily_spearman(
        volume_panel,
        left_column,
        right_column,
    )

    summary = daily_correlation.describe(
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    ).to_dict()

    summary["comparison"] = comparison
    similarity_rows.append(summary)

abnormal_volume_similarity_summary = (
    pd.DataFrame(similarity_rows)
    .set_index("comparison")
)

abnormal_volume_similarity_summary

,count,mean,std,min,5%,25%,50%,75%,95%,max
comparison,,,,,,,,,,
Abnormal volume vs dollar-volume level,2870.0,0.311864,0.120353,-0.101423,0.112430,0.229812,0.312134,0.394113,0.506707,0.702919
Abnormal volume vs momentum,2639.0,0.013088,0.144997,-0.558306,-0.216791,-0.086386,0.010328,0.109013,0.260181,0.538278
Abnormal volume vs realised volatility,2828.0,-0.030741,0.153050,-0.564045,-0.284604,-0.130342,-0.027211,0.067472,0.218739,0.533630
Volume–momentum interaction vs momentum,2639.0,-0.096441,0.102404,-0.418912,-0.267122,-0.164607,-0.096153,-0.025217,0.071094,0.318194


## 4. Predictive diagnostics

### 4.1 IC utilities

In [12]:
def calculate_daily_ic(
    panel,
    signal_column,
    return_column,
    minimum_stocks=20,
):
    results = []

    for date, group in panel.groupby("date", sort=True):
        valid = group[[signal_column, return_column]].dropna()

        if len(valid) < minimum_stocks:
            continue

        if valid[signal_column].nunique() < 2 or valid[return_column].nunique() < 2:
            continue

        correlation = valid[signal_column].corr(
            valid[return_column],
            method="spearman",
        )

        results.append(
            {
                "date": date,
                "ic": correlation,
            }
        )

    if not results:
        return pd.Series(dtype=float, name="ic")

    return pd.DataFrame(results).set_index("date")["ic"].rename("ic")


def summarise_ic(ic_series):
    ic_series = ic_series.dropna()

    count = len(ic_series)
    mean_ic = ic_series.mean()
    std_ic = ic_series.std()

    return pd.Series(
        {
            "count": count,
            "mean_ic": mean_ic,
            "std_ic": std_ic,
            "ic_ir": (mean_ic / std_ic if std_ic > 0 else np.nan),
            "t_stat": (
                mean_ic / std_ic * np.sqrt(count)
                if std_ic > 0 and count > 0
                else np.nan
            ),
            "positive_fraction": ((ic_series > 0).mean() if count > 0 else np.nan),
        }
    )


### 4.2 Horizon IC

In [13]:
volume_horizon_specifications = {
    "Abnormal volume — raw — 1-day": (
        "abnormal_volume_21_z",
        "forward_ret_1d",
    ),
    "Abnormal volume — raw — 5-day": (
        "abnormal_volume_21_z",
        "forward_ret_5d",
    ),
    "Abnormal volume — sector neutral — 1-day": (
        "abnormal_volume_21_sector_neutral_z",
        "forward_ret_1d",
    ),
    "Abnormal volume — sector neutral — 5-day": (
        "abnormal_volume_21_sector_neutral_z",
        "forward_ret_5d",
    ),
    "Interaction — raw — 1-day": (
        "volume_momentum_interaction_z",
        "forward_ret_1d",
    ),
    "Interaction — raw — 5-day": (
        "volume_momentum_interaction_z",
        "forward_ret_5d",
    ),
    "Interaction — sector neutral — 1-day": (
        "volume_momentum_interaction_sector_neutral_z",
        "forward_ret_1d",
    ),
    "Interaction — sector neutral — 5-day": (
        "volume_momentum_interaction_sector_neutral_z",
        "forward_ret_5d",
    ),
}

volume_horizon_ic_series = {}
volume_horizon_rows = []

for test_name, (
    signal_column,
    return_column,
) in volume_horizon_specifications.items():

    daily_ic = calculate_daily_ic(
        volume_panel,
        signal_column,
        return_column,
    )

    volume_horizon_ic_series[test_name] = daily_ic

    summary = summarise_ic(daily_ic)
    summary.name = test_name
    volume_horizon_rows.append(summary)

volume_horizon_ic_summary = pd.DataFrame(
    volume_horizon_rows
)

volume_horizon_ic_summary

,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
Abnormal volume — raw — 1-day,2869.0,0.003220,0.135292,0.023803,1.274949,0.512374
Abnormal volume — raw — 5-day,2865.0,-0.001302,0.133211,-0.009774,-0.523181,0.500524
Abnormal volume — sector neutral — 1-day,2869.0,0.003911,0.106630,0.036675,1.964432,0.516905
Abnormal volume — sector neutral — 5-day,2865.0,-0.000277,0.104216,-0.002658,-0.142278,0.495637
Interaction — raw — 1-day,2638.0,-0.004260,0.122033,-0.034911,-1.793101,0.485216
Interaction — raw — 5-day,2634.0,-0.004149,0.119998,-0.034571,-1.774296,0.491648
Interaction — sector neutral — 1-day,2638.0,-0.001583,0.103585,-0.015279,-0.784751,0.484837
Interaction — sector neutral — 5-day,2634.0,-0.000748,0.102175,-0.007321,-0.375712,0.487092


### 4.3 Subperiod stability

In [14]:
subperiods = {
    "2015–2018": ("2015-01-01", "2018-12-31"),
    "2019–2022": ("2019-01-01", "2022-12-31"),
    "2023–present": ("2023-01-01", None),
}

volume_subperiod_rows = []

subperiod_signal_specifications = {
    "Abnormal volume — raw": (
        "abnormal_volume_21_z"
    ),
    "Abnormal volume — sector neutral": (
        "abnormal_volume_21_sector_neutral_z"
    ),
    "Interaction — raw": (
        "volume_momentum_interaction_z"
    ),
    "Interaction — sector neutral": (
        "volume_momentum_interaction_sector_neutral_z"
    ),
}

for signal_name, signal_column in (
    subperiod_signal_specifications.items()
):
    for period_name, (
        start_date,
        end_date,
    ) in subperiods.items():

        period_mask = (
            volume_panel["date"]
            >= pd.Timestamp(start_date)
        )

        if end_date is not None:
            period_mask &= (
                volume_panel["date"]
                <= pd.Timestamp(end_date)
            )

        period_panel = volume_panel.loc[period_mask]

        period_ic = calculate_daily_ic(
            period_panel,
            signal_column,
            "forward_ret_5d",
        )

        summary = summarise_ic(period_ic).to_dict()
        summary["signal"] = signal_name
        summary["subperiod"] = period_name

        volume_subperiod_rows.append(summary)

volume_subperiod_ic_summary = (
    pd.DataFrame(volume_subperiod_rows)
    .set_index(["signal", "subperiod"])
)

volume_subperiod_ic_summary

count   mean_ic    std_ic  \
signal                           subperiod                                  
Abnormal volume — raw            2015–2018      985.0 -0.007907  0.131354   
                                 2019–2022     1008.0 -0.000099  0.139206   
                                 2023–present   872.0  0.004769  0.127926   
Abnormal volume — sector neutral 2015–2018      985.0 -0.006779  0.103169   
                                 2019–2022     1008.0  0.003379  0.106501   
                                 2023–present   872.0  0.002841  0.102476   
Interaction — raw                2015–2018      754.0 -0.007052  0.125101   
                                 2019–2022     1008.0  0.000269  0.113604   
                                 2023–present   872.0 -0.006745  0.122645   
Interaction — sector neutral     2015–2018      754.0 -0.001434  0.106429   
                                 2019–2022     1008.0  0.000510  0.098474   
                                 2023–present   872.0 -0.001609  0.102723   

                                                  ic_ir    t_stat  \
signal                           subperiod                          
Abnormal volume — raw            2015–2018    -0.060200 -1.889351   
                                 2019–2022    -0.000714 -0.022670   
                                 2023–present  0.037280  1.100872   
Abnormal volume — sector neutral 2015–2018    -0.065707 -2.062189   
                                 2019–2022     0.031727  1.007300   
                                 2023–present  0.027726  0.818731   
Interaction — raw                2015–2018    -0.056370 -1.547872   
                                 2019–2022     0.002371  0.075284   
                                 2023–present -0.054995 -1.623995   
Interaction — sector neutral     2015–2018    -0.013478 -0.370101   
                                 2019–2022     0.005183  0.164561   
                                 2023–present -0.015664 -0.462548   

                                               positive_fraction  
signal                           subperiod                        
Abnormal volume — raw            2015–2018              0.480203  
                                 2019–2022              0.499008  
                                 2023–present           0.525229  
Abnormal volume — sector neutral 2015–2018              0.473096  
                                 2019–2022              0.504960  
                                 2023–present           0.510321  
Interaction — raw                2015–2018              0.482759  
                                 2019–2022              0.498016  
                                 2023–present           0.491972  
Interaction — sector neutral     2015–2018              0.494695  
                                 2019–2022              0.479167  
                                 2023–present           0.489679

### 4.4 Non-overlapping IC

In [15]:
available_dates = np.array(
    sorted(volume_panel["date"].dropna().unique())
)

non_overlapping_signal_specifications = {
    "Abnormal volume — raw": (
        "abnormal_volume_21_z"
    ),
    "Abnormal volume — sector neutral": (
        "abnormal_volume_21_sector_neutral_z"
    ),
    "Interaction — raw": (
        "volume_momentum_interaction_z"
    ),
    "Interaction — sector neutral": (
        "volume_momentum_interaction_sector_neutral_z"
    ),
}

volume_non_overlapping_rows = []

for signal_name, signal_column in (
    non_overlapping_signal_specifications.items()
):
    for offset in range(5):
        selected_dates = available_dates[offset::5]

        offset_panel = volume_panel.loc[
            volume_panel["date"].isin(selected_dates)
        ]

        offset_ic = calculate_daily_ic(
            offset_panel,
            signal_column,
            "forward_ret_5d",
        )

        summary = summarise_ic(offset_ic).to_dict()
        summary["signal"] = signal_name
        summary["offset"] = offset

        volume_non_overlapping_rows.append(summary)

volume_non_overlapping_ic = (
    pd.DataFrame(volume_non_overlapping_rows)
    .set_index(["signal", "offset"])
)

volume_non_overlapping_ic

count   mean_ic    std_ic     ic_ir  \
signal                           offset                                        
Abnormal volume — raw            0       573.0 -0.000670  0.137559 -0.004872   
                                 1       573.0 -0.001474  0.136563 -0.010794   
                                 2       573.0 -0.003865  0.130726 -0.029567   
                                 3       573.0 -0.003016  0.129858 -0.023225   
                                 4       573.0  0.002515  0.131535  0.019120   
Abnormal volume — sector neutral 0       573.0 -0.000772  0.106908 -0.007223   
                                 1       573.0  0.002501  0.100571  0.024868   
                                 2       573.0  0.000059  0.105656  0.000563   
                                 3       573.0 -0.003811  0.104095 -0.036610   
                                 4       573.0  0.000638  0.104002  0.006130   
Interaction — raw                0       527.0 -0.003781  0.119960 -0.031517   
                                 1       526.0  0.002508  0.117029  0.021434   
                                 2       527.0 -0.003830  0.118764 -0.032245   
                                 3       527.0 -0.009604  0.124115 -0.077383   
                                 4       527.0 -0.006024  0.120134 -0.050140   
Interaction — sector neutral     0       527.0 -0.001313  0.100807 -0.013027   
                                 1       526.0  0.002387  0.096623  0.024701   
                                 2       527.0  0.001507  0.104762  0.014381   
                                 3       527.0 -0.003073  0.108430 -0.028338   
                                 4       527.0 -0.003241  0.100093 -0.032382   

                                           t_stat  positive_fraction  
signal                           offset                               
Abnormal volume — raw            0      -0.116617           0.520070  
                                 1      -0.258388           0.507853  
                                 2      -0.707749           0.486911  
                                 3      -0.555941           0.490401  
                                 4       0.457692           0.497382  
Abnormal volume — sector neutral 0      -0.172900           0.495637  
                                 1       0.595270           0.493892  
                                 2       0.013480           0.497382  
                                 3      -0.876352           0.499127  
                                 4       0.146740           0.492147  
Interaction — raw                0      -0.723528           0.481973  
                                 1       0.491584           0.530418  
                                 2      -0.740227           0.491461  
                                 3      -1.776447           0.478178  
                                 4      -1.151045           0.476281  
Interaction — sector neutral     0      -0.299061           0.476281  
                                 1       0.566500           0.517110  
                                 2       0.330137           0.497154  
                                 3      -0.650551           0.468691  
                                 4      -0.743371           0.476281

In [16]:
volume_non_overlapping_consistency = (
    volume_non_overlapping_ic
    .reset_index()
    .groupby("signal")
    .agg(
        mean_ic_across_offsets=("mean_ic", "mean"),
        min_mean_ic=("mean_ic", "min"),
        max_mean_ic=("mean_ic", "max"),
        mean_t_stat=("t_stat", "mean"),
        positive_offsets=(
            "mean_ic",
            lambda values: int((values > 0).sum()),
        ),
    )
)

volume_non_overlapping_consistency

,mean_ic_across_offsets,min_mean_ic,max_mean_ic,mean_t_stat,positive_offsets
signal,,,,,
Abnormal volume — raw,-0.001302,-0.003865,0.002515,-0.236201,1
Abnormal volume — sector neutral,-0.000277,-0.003811,0.002501,-0.058753,3
Interaction — raw,-0.004146,-0.009604,0.002508,-0.779933,1
Interaction — sector neutral,-0.000747,-0.003241,0.002387,-0.159269,2


### 4.5 Incremental information beyond momentum

In [17]:
volume_panel = volume_panel.reset_index(drop=True).copy()

volume_panel["volume_momentum_interaction_incremental"] = np.nan

for _, group_indices in volume_panel.groupby("date", sort=False).groups.items():
    valid_indices = (
        volume_panel.loc[
            group_indices,
            [
                "volume_momentum_interaction_z",
                "mom_12_1m_z",
            ],
        ]
        .dropna()
        .index
    )

    if len(valid_indices) < 20:
        continue

    interaction = volume_panel.loc[
        valid_indices,
        "volume_momentum_interaction_z",
    ].to_numpy()

    momentum = volume_panel.loc[
        valid_indices,
        "mom_12_1m_z",
    ].to_numpy()

    if np.isclose(np.std(momentum), 0):
        continue

    design_matrix = np.column_stack(
        [
            np.ones(len(momentum)),
            momentum,
        ]
    )

    coefficients = np.linalg.lstsq(
        design_matrix,
        interaction,
        rcond=None,
    )[0]

    incremental_interaction = interaction - design_matrix @ coefficients

    volume_panel.loc[
        valid_indices,
        "volume_momentum_interaction_incremental",
    ] = incremental_interaction

In [19]:
def cross_sectional_zscore(series):
    valid = series.dropna()

    if len(valid) < 2:
        return pd.Series(
            np.nan,
            index=series.index,
            dtype=float,
        )

    standard_deviation = valid.std(ddof=0)

    if (
        not np.isfinite(standard_deviation)
        or np.isclose(standard_deviation, 0)
    ):
        return pd.Series(
            np.nan,
            index=series.index,
            dtype=float,
        )

    return (
        series - valid.mean()
    ) / standard_deviation

In [20]:
volume_panel[
    "volume_momentum_interaction_incremental_z"
] = (
    volume_panel
    .groupby("date")[
        "volume_momentum_interaction_incremental"
    ]
    .transform(cross_sectional_zscore)
)

In [21]:
incremental_interaction_correlations = []

orthogonality_sample = volume_panel.dropna(
    subset=[
        "volume_momentum_interaction_incremental_z",
        "mom_12_1m_z",
    ]
)

for date, group in orthogonality_sample.groupby(
    "date",
    sort=True,
):
    correlation = group[
        "volume_momentum_interaction_incremental_z"
    ].corr(
        group["mom_12_1m_z"],
        method="pearson",
    )

    incremental_interaction_correlations.append(
        {
            "date": date,
            "correlation": correlation,
        }
    )

incremental_interaction_orthogonality = (
    pd.DataFrame(incremental_interaction_correlations)
    ["correlation"]
    .describe()
)

incremental_interaction_orthogonality

count    2.639000e+03
mean     2.051385e-18
std      1.026337e-16
min     -7.229755e-16
25%     -4.624153e-17
50%      3.268111e-18
75%      5.076584e-17
max      7.166418e-16
Name: correlation, dtype: float64

In [22]:
interaction_common_sample = volume_panel.dropna(
    subset=[
        "mom_12_1m_z",
        "volume_momentum_interaction_z",
        "volume_momentum_interaction_incremental_z",
        "forward_ret_1d",
        "forward_ret_5d",
    ]
)

incremental_interaction_rows = []

incremental_signal_specifications = {
    "Raw 12–1 momentum": "mom_12_1m_z",
    "Volume–momentum interaction": (
        "volume_momentum_interaction_z"
    ),
    "Interaction incremental": (
        "volume_momentum_interaction_incremental_z"
    ),
}

for signal_name, signal_column in (
    incremental_signal_specifications.items()
):
    for horizon_name, return_column in {
        "1-day": "forward_ret_1d",
        "5-day": "forward_ret_5d",
    }.items():

        daily_ic = calculate_daily_ic(
            interaction_common_sample,
            signal_column,
            return_column,
        )

        summary = summarise_ic(daily_ic).to_dict()
        summary["signal"] = signal_name
        summary["horizon"] = horizon_name

        incremental_interaction_rows.append(summary)

volume_interaction_incremental_ic_summary = (
    pd.DataFrame(incremental_interaction_rows)
    .set_index(["signal", "horizon"])
)

volume_interaction_incremental_ic_summary

count   mean_ic    std_ic     ic_ir  \
signal                      horizon                                         
Raw 12–1 momentum           1-day    2634.0  0.019591  0.282763  0.069284   
                            5-day    2634.0  0.019718  0.277559  0.071042   
Volume–momentum interaction 1-day    2634.0 -0.004471  0.121998 -0.036646   
                            5-day    2634.0 -0.004149  0.119998 -0.034571   
Interaction incremental     1-day    2634.0 -0.002574  0.122632 -0.020987   
                            5-day    2634.0 -0.002751  0.120063 -0.022911   

                                       t_stat  positive_fraction  
signal                      horizon                               
Raw 12–1 momentum           1-day    3.555848           0.541762  
                            5-day    3.646044           0.555809  
Volume–momentum interaction 1-day   -1.880745           0.484434  
                            5-day   -1.774296           0.491648  
Interaction incremental     1-day   -1.077126           0.488610  
                            5-day   -1.175870           0.491648

## 5. Conclusion

The abnormal-volume family passes the construction and data-quality checks but does not qualify for promotion as a core factor.

### 5.1 Abnormal volume

The 21-day abnormal-volume signal is constructed without lookahead and has 99.3% coverage. It is largely distinct from conventional momentum and realised volatility, but this distinctiveness does not translate into predictive value:

- Raw 5-day IC is -0.0013 ($t=-0.52$).
- Sector-neutral 5-day IC is -0.0003 ($t=-0.14$).
- Subperiod IC changes sign and remains economically small.
- Only one of five raw non-overlapping offsets is positive; the sector-neutral offsets are mixed and individually insignificant.

The mildly positive sector-neutral 1-day IC of 0.0039 ($t=1.96$) does not persist at the project’s primary 5-day horizon and is insufficient to justify promotion.

### 5.2 Volume–momentum interaction

The interaction between abnormal volume and 12–1 momentum also fails:

- Raw 5-day IC is -0.0041 ($t=-1.77$).
- Sector-neutral 5-day IC is -0.0007 ($t=-0.38$).
- Subperiod and non-overlapping results are unstable and predominantly non-positive.

Cross-sectional residualisation successfully removes the interaction’s linear relationship with conventional momentum: its daily Pearson correlation with momentum is approximately zero after residualisation.

However, the orthogonal component does not provide complementary predictive information:

- Incremental 1-day IC is -0.0026 ($t=-1.08$).
- Incremental 5-day IC is -0.0028 ($t=-1.18$).

The negative estimates are too small and statistically weak to support treating the inverted interaction as a new predictor.

**Decision:** reject both abnormal volume and the volume–momentum interaction. Do not proceed to portfolio-level backtesting for either signal.

### 5.3 Research lesson

Signal distinctiveness and an intuitive economic narrative are not sufficient promotion criteria. A candidate must also demonstrate stable predictive information across horizons, subperiods, non-overlapping samples, and incremental tests.

This completes the planned standalone factor-discovery stage. The retained core signals remain:

1. **12–1 momentum**
2. **Realised volatility**

The next stage will combine these signals within a constrained multi-factor portfolio and evaluate diversification, exposures, turnover, transaction costs, and robustness.